# 12_Gradio_Endpoint_Test.ipynb

## Purpose

This notebook provides a simple **Gradio user interface** to test the deployed AWS SageMaker endpoint:

`heart-attack-team05-s502-endpoint`

It does **not** retrain or redeploy the model.

It reads the expected feature list from the project metadata and sends the entered values to the live SageMaker endpoint.

> Educational prototype only. The output describes similarity to respondents who reported a previous heart attack. It does not predict a future heart attack and is not a medical diagnosis.

## 1. Install Gradio

In [2]:
# ============================================================
# INSTALL GRADIO DEPENDENCIES
# ============================================================

!pip install -q "starlette<1" "fastapi<1" --upgrade
!pip install -q gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.52.1 which is incompatible.


## 2. Imports and AWS configuration

In [3]:
import json
from pathlib import Path

import boto3
import gradio as gr
import pandas as pd


REGION = "ap-southeast-1"

ENDPOINT_NAME = (
    "heart-attack-team05-s502-endpoint"
)

runtime = boto3.client(
    "sagemaker-runtime",
    region_name=REGION,
)

sm_client = boto3.client(
    "sagemaker",
    region_name=REGION,
)

print("Region   :", REGION)
print("Endpoint :", ENDPOINT_NAME)

Region   : ap-southeast-1
Endpoint : heart-attack-team05-s502-endpoint


## 3. Verify that the SageMaker endpoint is available

In [4]:
endpoint_desc = sm_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)

endpoint_status = endpoint_desc[
    "EndpointStatus"
]

print("Endpoint status:", endpoint_status)

if endpoint_status != "InService":
    raise RuntimeError(
        f"Endpoint is not ready. "
        f"Current status: {endpoint_status}"
    )

print("✅ Endpoint is ready.")

Endpoint status: InService
✅ Endpoint is ready.


## 4. Locate project files

In [5]:
PROJECT_ROOT = Path(
    "/home/sagemaker-user/"
    "Heart_Attack_Risk_Assessment"
)

TRAIN_DATA = (
    PROJECT_ROOT
    / "data"
    / "full_train_raw.csv"
)

METADATA_FILE = (
    PROJECT_ROOT
    / "artifacts"
    / "stage10_pipeline"
    / "metadata_template.json"
)

if not TRAIN_DATA.exists():
    raise FileNotFoundError(
        f"Training-format data not found: "
        f"{TRAIN_DATA}"
    )

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Metadata file not found: "
        f"{METADATA_FILE}"
    )

print("Training data :", TRAIN_DATA)
print("Metadata      :", METADATA_FILE)
print("✅ Project files found.")

Training data : /home/sagemaker-user/Heart_Attack_Risk_Assessment/data/full_train_raw.csv
Metadata      : /home/sagemaker-user/Heart_Attack_Risk_Assessment/artifacts/stage10_pipeline/metadata_template.json
✅ Project files found.


## 5. Load the expected feature contract

In [6]:
with open(
    METADATA_FILE,
    "r",
    encoding="utf-8",
) as f:
    metadata = json.load(f)

FEATURES = metadata["features"]

DISCLAIMER = metadata.get(
    "disclaimer",
    (
        "Educational prototype only. "
        "This is not a medical diagnosis."
    ),
)

df = pd.read_csv(
    TRAIN_DATA
)

missing_features = [
    feature
    for feature in FEATURES
    if feature not in df.columns
]

if missing_features:
    raise RuntimeError(
        "Training-format data is missing "
        f"expected features: {missing_features}"
    )

print("Feature count:", len(FEATURES))
print("Dataset rows :", len(df))
print("✅ Feature contract loaded.")

Feature count: 39
Dataset rows : 353653
✅ Feature contract loaded.


## 6. Endpoint invocation helper

In [7]:
def invoke_endpoint(record):

    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(
            record
        ).encode("utf-8"),
    )

    body = (
        response["Body"]
        .read()
        .decode("utf-8")
    )

    return json.loads(body)


print("✅ Endpoint helper ready.")

✅ Endpoint helper ready.


## 7. Build Gradio input components dynamically

In [8]:
def make_component(feature):

    series = (
        df[feature]
        .dropna()
    )

    if pd.api.types.is_numeric_dtype(
        series
    ):

        unique_values = sorted(
            series.unique().tolist()
        )

        # Small discrete numeric set -> dropdown
        if len(unique_values) <= 15:

            return gr.Dropdown(
                choices=unique_values,
                value=(
                    unique_values[0]
                    if unique_values
                    else None
                ),
                label=feature,
            )

        # Larger / continuous numeric -> number input
        default_value = (
            float(series.median())
            if len(series)
            else 0
        )

        return gr.Number(
            value=default_value,
            label=feature,
        )

    unique_values = sorted(
        series
        .astype(str)
        .unique()
        .tolist()
    )

    return gr.Dropdown(
        choices=unique_values,
        value=(
            unique_values[0]
            if unique_values
            else None
        ),
        label=feature,
    )


components = {
    feature:
        make_component(feature)
    for feature in FEATURES
}

print(
    "✅ Components prepared for",
    len(components),
    "features."
)

✅ Components prepared for 39 features.


## 8. Prediction and example-loading functions

In [9]:
def build_record(*values):

    record = {}

    for feature, value in zip(
        FEATURES,
        values,
    ):

        if value is None:
            record[feature] = None

        elif hasattr(
            value,
            "item",
        ):
            record[feature] = value.item()

        else:
            record[feature] = value

    return record


def predict(*values):

    try:

        record = build_record(
            *values
        )

        result = invoke_endpoint(
            record
        )

        score = result.get(
            "association_score"
        )

        threshold = result.get(
            "threshold"
        )

        positive = result.get(
            "positive_class"
        )

        classification = result.get(
            "classification"
        )

        model_version = result.get(
            "model_package_version"
        )

        result_text = (
            f"Association score: {score}\n\n"
            f"Threshold: {threshold}\n\n"
            f"Positive class: {positive}\n\n"
            f"Classification:\n"
            f"{classification}\n\n"
            f"Model Registry version: "
            f"{model_version}"
        )

        return (
            result_text,
            result,
            record,
        )

    except Exception as e:

        error = {
            "error": str(e)
        }

        return (
            f"Prediction failed:\n{e}",
            error,
            {},
        )


def load_random_example():

    row = (
        df[FEATURES]
        .sample(n=1)
        .iloc[0]
    )

    values = []

    for feature in FEATURES:

        value = row[feature]

        if pd.isna(value):
            value = None

        elif hasattr(
            value,
            "item",
        ):
            value = value.item()

        values.append(value)

    return values


print("✅ Prediction functions ready.")

✅ Prediction functions ready.


## 9. Create and launch the Gradio app

In [10]:
with gr.Blocks(
    title=(
        "Heart Attack History "
        "Association Demo"
    )
) as demo:

    gr.Markdown(
        '''
# Heart Attack History Association Demo

This interface calls the deployed **AWS SageMaker endpoint**.

The model estimates how similar the entered profile is to respondents who reported a previous heart attack in the training data.

**Educational prototype only.**
This is not a diagnosis and does not predict a future heart attack.
'''
    )

    with gr.Accordion(
        "Respondent Features",
        open=True,
    ):

        input_components = []

        for i in range(
            0,
            len(FEATURES),
            2,
        ):

            with gr.Row():

                feature1 = FEATURES[i]

                comp1 = components[
                    feature1
                ]

                comp1.render()

                input_components.append(
                    comp1
                )

                if (
                    i + 1
                    < len(FEATURES)
                ):

                    feature2 = (
                        FEATURES[
                            i + 1
                        ]
                    )

                    comp2 = components[
                        feature2
                    ]

                    comp2.render()

                    input_components.append(
                        comp2
                    )

    with gr.Row():

        predict_button = gr.Button(
            "Run Prediction",
            variant="primary",
        )

        example_button = gr.Button(
            "Load Random Example"
        )

    gr.Markdown(
        "## Prediction Result"
    )

    result_text = gr.Textbox(
        label="Result",
        lines=9,
        interactive=False,
    )

    with gr.Accordion(
        "Raw Endpoint Response",
        open=False,
    ):

        raw_json = gr.JSON(
            label="SageMaker response"
        )

    with gr.Accordion(
        "JSON Sent to Endpoint",
        open=False,
    ):

        request_json = gr.JSON(
            label="Request payload"
        )

    gr.Markdown(
        f'''
---

### Disclaimer

{DISCLAIMER}
'''
    )

    predict_button.click(
        fn=predict,
        inputs=input_components,
        outputs=[
            result_text,
            raw_json,
            request_json,
        ],
    )

    example_button.click(
        fn=load_random_example,
        inputs=[],
        outputs=input_components,
    )


demo.launch(
    inline=True,
    share=True,
    debug=True
)

* Running on local URL:  http://127.0.0.1:7860


* Running on public URL: https://60129661396486eb34.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
try:
    demo.close()
    print("demo.close() called.")
except Exception as e:
    print("demo.close():", e)

In [ ]:
# ============================================================
# SAFE GRADIO CLEANUP
#
# Stops the Gradio app and releases its local server/port.
#
# DOES NOT:
#   - delete the SageMaker endpoint
#   - delete the endpoint configuration
#   - delete the SageMaker model
#   - change Model Registry
#   - change Notebook 10/11 resources
# ============================================================

print("=" * 70)
print("GRADIO CLEANUP")
print("=" * 70)

try:
    demo.close()

    print("✅ Gradio app stopped.")
    print("✅ Local Gradio server/port released.")

except NameError:
    print(
        "ℹ️ Variable 'demo' does not exist in this kernel."
    )
    print(
        "The Gradio app may already be stopped "
        "or the kernel may have been restarted."
    )

except Exception as e:
    print("⚠️ Gradio cleanup returned:")
    print(e)

print()
print("SageMaker endpoint was NOT changed.")
print("Endpoint: heart-attack-team05-s502-endpoint")
print("=" * 70)

## 10. Notes

- Keep the SageMaker endpoint `InService` while using this app.
- This notebook only invokes the endpoint; it does not change the deployed model.
- `Load Random Example` fills the form with one row from the project training-format dataset.
- `Run Prediction` sends the current form values to SageMaker.
- The raw request and raw endpoint response are shown for demonstration and debugging.